# Src 02: Review & Approve Inventory (REQUIRED)

## Overview

This is a **required manual step** that must be run in Databricks UI.

**What this step does:**
1. Load inventory from Step 1
2. View statistics and identify issues
3. Apply filters to remove unwanted dashboards
4. Type CONFIRM to upload approved inventory
5. Verify upload succeeded

**Requirements:**
- Must be run interactively in Databricks UI
- Cannot be automated via CLI/jobs
- Requires manual review and confirmation

---

## Cell 1: Configuration Setup

In [ ]:
# Path resolution: add parent of notebooks dir (= src/) to sys.path so helpers can be imported
import sys
import os

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    if not notebook_path.startswith('/Workspace'):
        notebook_path = '/Workspace' + notebook_path
    notebook_dir = os.path.dirname(notebook_path)
    bundle_parent = os.path.dirname(notebook_dir)
    if bundle_parent not in sys.path:
        sys.path.insert(0, bundle_parent)
    print(f"Added to sys.path: {bundle_parent}")
except Exception:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
# ============================================================================
# CONFIGURATION - UPDATE THESE VALUES FOR YOUR ENVIRONMENT
# ============================================================================

# Option 1: Use widgets (for interactive execution)
# These will be populated from databricks.yml when run as a job,
# or you can set them manually for interactive use

try:
    # Try to get from widgets (set by job parameters)
    dbutils.widgets.text("volume_base", "/Volumes/YOUR_CATALOG/YOUR_SCHEMA/dashboard_migration")
    dbutils.widgets.text("catalog", "YOUR_CATALOG")
    
    volume_base = dbutils.widgets.get("volume_base")
    catalog = dbutils.widgets.get("catalog")
except:
    # Fallback defaults - UPDATE THESE for your environment
    volume_base = "/Volumes/YOUR_CATALOG/YOUR_SCHEMA/dashboard_migration"
    catalog = "YOUR_CATALOG"

# ============================================================================
# DERIVED PATHS (do not modify)
# ============================================================================
INVENTORY_SOURCE = f"{volume_base}/dashboard_inventory/inventory.csv"
INVENTORY_APPROVED = f"{volume_base}/dashboard_inventory_approved/inventory_approved.csv"

# ============================================================================
# DISPLAY CONFIGURATION
# ============================================================================
print("=" * 70)
print("STEP 2: REVIEW & APPROVE INVENTORY")
print("=" * 70)
print()
print(f"📁 Volume Base:      {volume_base}")
print(f"📚 Catalog:          {catalog}")
print()
print(f"📄 Source Inventory: {INVENTORY_SOURCE}")
print(f"✅ Approved Output:  {INVENTORY_APPROVED}")
print()
print("=" * 70)

# Validate paths exist
if "YOUR_CATALOG" in volume_base or "YOUR_SCHEMA" in volume_base:
    print("⚠️  WARNING: Update volume_base with your actual catalog/schema values!")
    print("   Edit this cell and replace YOUR_CATALOG and YOUR_SCHEMA")


## Cell 1.5: Configuration Check

**Quick check:** Verify volume paths are accessible before loading inventory.

In [ ]:
# Quick volume path check
print("🔍 Checking volume paths...")
print()

try:
    # Check if source inventory path exists
    files = dbutils.fs.ls(volume_base)
    print(f"✅ Volume base accessible: {volume_base}")
    
    # Check if source inventory file exists
    inventory_files = dbutils.fs.ls(f"{volume_base}/dashboard_inventory")
    print(f"✅ Source inventory directory accessible")
    
    # Check if target approved directory is accessible (may not exist yet)
    try:
        dbutils.fs.ls(f"{volume_base}/dashboard_inventory_approved")
        print(f"✅ Approved inventory directory exists")
    except:
        print(f"ℹ️  Approved inventory directory will be created when you upload")
    
    print()
    print("✅ Configuration check passed")
    print()
    
except Exception as e:
    print(f"❌ Configuration check failed: {e}")
    print()
    print("Please verify:")
    print(f"  1. Volume path is correct: {volume_base}")
    print(f"  2. You have access to the volume")
    print(f"  3. Src_01_Inventory_Generation has been run successfully")
    print()
    raise

## Cell 2: Load Inventory

In [ ]:
# Load inventory CSV
print("📥 Loading inventory...\n")

df = spark.read.csv(INVENTORY_SOURCE, header=True, inferSchema=True)

print(f"✅ Loaded {df.count()} dashboards\n")
print("="*70)
print("INVENTORY SUMMARY")
print("="*70)

# Summary statistics
print(f"\n📊 Total Dashboards: {df.count()}")
print(f"\n🎯 Activity Level Distribution:")
df.groupBy('activity_level').count().orderBy('count', ascending=False).show()

print(f"\n🔧 Complexity Distribution:")
df.groupBy('complexity').count().orderBy('count', ascending=False).show()

print(f"\n📈 Table Usage Stats:")
df.select('table_count').summary('min', 'max', 'mean').show()

print("\n" + "="*70)

## Cell 3: View Full Inventory

Review all dashboards - scroll through to identify any issues

In [ ]:
# Display full inventory for review
print("📋 Full Inventory (sorted by table count):\n")
display(df.orderBy('table_count', ascending=False))

## Cell 4: Identify Issues

Check for dashboards with potential problems

In [ ]:
# Check for dashboards with fallback names (API fetch failed)
print("⚠️  Dashboards with Fallback Names (API fetch failed):\n")
fallback_names = df.filter(df.dashboard_name.startswith('Dashboard_'))
fallback_count = fallback_names.count()

if fallback_count > 0:
    print(f"Found {fallback_count} dashboards with fallback names:")
    display(fallback_names.select('dashboard_id', 'dashboard_name', 'table_count', 'activity_level'))
    print("\n💡 These dashboards likely no longer exist or are inaccessible.")
    print("   Recommendation: EXCLUDE from migration\n")
else:
    print("✅ All dashboards have real names\n")

# Check for inactive dashboards
print("\n📊 Inactive Dashboards (no recent usage):\n")
inactive = df.filter(df.activity_level == 'Inactive')
inactive_count = inactive.count()

if inactive_count > 0:
    print(f"Found {inactive_count} inactive dashboards:")
    display(inactive.select('dashboard_id', 'dashboard_name', 'table_count', 'last_accessed'))
    print("\n💡 Consider excluding these from migration if unused\n")
else:
    print("✅ All dashboards have recent activity\n")

# Check for dashboards with no table references
print("\n📊 Dashboards with Zero Tables:\n")
no_tables = df.filter(df.table_count == 0)
no_tables_count = no_tables.count()

if no_tables_count > 0:
    print(f"Found {no_tables_count} dashboards with no table references:")
    display(no_tables.select('dashboard_id', 'dashboard_name', 'activity_level'))
    print("\n💡 These may be text-only dashboards or have issues\n")
else:
    print("✅ All dashboards reference tables\n")

## Cell 5: Filter & Analyze

**CUSTOMIZE THIS CELL** with your filtering criteria.

Uncomment/modify filters as needed.

In [ ]:
# ============================================================================
# FILTERING LOGIC - Customize as needed
# ============================================================================

print("🎯 Applying QC filters...\n")

# Start with full inventory
df_approved = df

# Filter 1: Remove dashboards with fallback names (failed API lookups)
print("Filter 1: Removing dashboards with fallback names...")
df_approved = df_approved.filter(~df_approved.dashboard_name.startswith('Dashboard_'))
print(f"  Remaining: {df_approved.count()}\n")

# Filter 2: Only include dashboards with table references
print("Filter 2: Requiring table_count > 0...")
df_approved = df_approved.filter(df_approved.table_count > 0)
print(f"  Remaining: {df_approved.count()}\n")

# Filter 3: Remove inactive dashboards (optional - uncomment to enable)
# print("Filter 3: Removing inactive dashboards...")
# df_approved = df_approved.filter(df_approved.activity_level != 'Inactive')
# print(f"  Remaining: {df_approved.count()}\n")

# Filter 4: Minimum activity level (uncomment to enable)
# print("Filter 4: Requiring minimum activity...")
# df_approved = df_approved.filter(df_approved.activity_level.isin(['Very Active', 'Active']))
# print(f"  Remaining: {df_approved.count()}\n")

# Filter 5: Custom dashboard IDs (uncomment and modify to enable)
# approved_ids = ['01efeeea51701a2c949822fad50b0981', '01efeef2957a1f0a82d4b82ac128a87f']
# print(f"Filter 5: Using custom ID list ({len(approved_ids)} IDs)...")
# df_approved = df_approved.filter(df_approved.dashboard_id.isin(approved_ids))
# print(f"  Remaining: {df_approved.count()}\n")

print("="*70)
print(f"✅ APPROVED: {df_approved.count()} dashboards")
print(f"❌ EXCLUDED: {df.count() - df_approved.count()} dashboards")
print("="*70)

## Cell 6: Review Analyzed Inventory

In [ ]:
# Display approved inventory for final review
print("📋 APPROVED INVENTORY FOR MIGRATION:\n")
display(df_approved.orderBy('table_count', ascending=False))

# Summary of approved dashboards
print("\n📊 Approved Dashboard Summary:")
print(f"  Total: {df_approved.count()}")
print(f"  Avg tables/dashboard: {df_approved.agg({'table_count': 'avg'}).collect()[0][0]:.1f}")
print(f"  Total unique tables: {df_approved.agg({'unique_tables': 'sum'}).collect()[0][0]}")
print(f"\n  Activity Levels:")
df_approved.groupBy('activity_level').count().orderBy('count', ascending=False).show(truncate=False)
print(f"\n  Complexity Levels:")
df_approved.groupBy('complexity').count().orderBy('count', ascending=False).show(truncate=False)

## Cell 7: Upload Approved Inventory

**⚠️ IMPORTANT:** 
- **UI Mode**: Requires typing 'CONFIRM' to upload
- **CLI Mode**: Auto-uploads without confirmation

In [ ]:
# UPLOAD APPROVED INVENTORY

print("="*70)
print("⚠️  UPLOAD CONFIRMATION REQUIRED")
print("="*70)

approved_pandas = df_approved.toPandas()
csv_content = approved_pandas.to_csv(index=False)

print(f"\n📊 Ready to upload {len(approved_pandas)} dashboards")
print(f"💾 Size: {len(csv_content) / 1024:.1f} KB")
print(f"📁 Destination: {INVENTORY_APPROVED}\n")

# Check if file already exists
try:
    existing = spark.read.csv(INVENTORY_APPROVED, header=True, inferSchema=True)
    existing_count = existing.count()
    print("="*70)
    print("⚠️  WARNING: EXISTING FILE WILL BE OVERWRITTEN")
    print("="*70)
    print(f"\n📋 Current approved inventory: {existing_count} dashboards")
    print(f"📋 New filtered inventory: {len(approved_pandas)} dashboards")
    print(f"\nIf you have manually uploaded a file to dashboard_inventory_approved/,")
    print(f"it will be REPLACED with the results from Cell 5-6 filters.")
    print(f"\nTyping CONFIRM means you ACCEPT this new filtered inventory")
    print(f"and discard any previously uploaded file.\n")
except:
    print("✅ No existing approved inventory found - this will be a fresh upload\n")

# Require explicit confirmation
print("="*70)
print("To proceed, type 'CONFIRM' (uppercase, exact spelling)")
print("To cancel, type anything else or press Ctrl+C")
print("="*70)
response = input("\n⚠️  Type 'CONFIRM' to accept and upload this filtered inventory: ")

if response.strip().upper() == 'CONFIRM':
    # ========================================================================
    # ARCHIVE OLD APPROVED INVENTORY (preserves audit trail)
    # ========================================================================
    print("\n" + "="*70)
    print("📦 ARCHIVING OLD APPROVED INVENTORY")
    print("="*70)
    
    try:
        from helpers import archive_old_files as archive_helper
        
        archive_result = archive_helper(
            source_folder=f"{volume_base}/dashboard_inventory_approved",
            file_pattern="inventory_approved.csv",
            archive_subfolder="archive",
            min_age_minutes=0  # Archive ALL files regardless of age
        )
        
        if 'error' in archive_result:
            print(f"\n⚠️  Archive warning: {archive_result['error']}")
        else:
            if archive_result['archived_count'] > 0:
                print(f"\n✅ Archived {archive_result['archived_count']} file(s)")
                print(f"📁 Archive location: {archive_result['archive_path']}")
            else:
                print(f"\nℹ️  No previous approved inventory to archive")
    except Exception as e:
        print(f"\n⚠️  Archive failed: {e}")
        print("   Continuing with upload...")
    
    print("="*70)
    
    # ========================================================================
    # UPLOAD NEW APPROVED INVENTORY
    # ========================================================================
    print("\n📤 Uploading new approved inventory...")
    dbutils.fs.put(INVENTORY_APPROVED, csv_content, overwrite=True)
    print("\n" + "="*70)
    print("✅ UPLOADED SUCCESSFULLY")
    print("="*70)
    print(f"📁 Location: {INVENTORY_APPROVED}")
    print(f"📊 Dashboards: {len(approved_pandas)}")
    print(f"\n🎯 Next: Run Cell 8 to verify upload")
else:
    print("\n" + "="*70)
    print(f"❌ UPLOAD CANCELLED (you typed: '{response}')")
    print("="*70)
    print("\nNo changes made to approved inventory.")
    print("To upload, re-run this cell and type: CONFIRM")

## Cell 8: Verify Upload

Verify the approved inventory was uploaded successfully and is ready for Step 2

In [ ]:
# VERIFY UPLOAD
print("\n🔍 Verifying upload...\n")

try:
    # Read back the uploaded file
    df_verify = spark.read.csv(INVENTORY_APPROVED, header=True, inferSchema=True)
    
    # Get file metadata
    file_info_list = dbutils.fs.ls(f"{volume_base}/dashboard_inventory_approved")
    
    for file in file_info_list:
        if file.name == 'inventory_approved.csv':
            from datetime import datetime
            modified_time = datetime.fromtimestamp(file.modificationTime / 1000)
            
            print("="*70)
            print("✅ VERIFICATION SUCCESSFUL")
            print("="*70)
            print(f"📁 Location: {INVENTORY_APPROVED}")
            print(f"📊 Dashboards: {df_verify.count()}")
            print(f"📅 Modified: {modified_time.strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"💾 Size: {file.size / 1024:.1f} KB")
            print("="*70)
            print("\n🎯 READY FOR STEP 2")
            print("   Run: databricks bundle run export_transform -t dev")
            print("="*70)
            break
except Exception as e:
    print("="*70)
    print("❌ VERIFICATION FAILED")
    print("="*70)
    print(f"Error: {e}")
    print("\nPossible reasons:")
    print("  - Upload was cancelled in Cell 7")
    print("  - File permissions issue")
    print("  - Volume path incorrect")
    print("\nTo fix: Re-run Cell 7 and type CONFIRM")
    print("="*70)